---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---

In [1]:
library(reticulate)

# Ensure the default reticulate virtual environment is used
use_virtualenv("r-reticulate", required = TRUE)

# List of Python packages to install or upgrade (excluding 're')
pkgs <- c(
  "numpy",
  "pandas",
  "streamlit",
  "python_dateutil",
  "tabulate",
  "swifter",
  "rpy2",
  "pyreadr",
  "papermill", # Add other necessary packages here
  "nbformat", # Add nbformat for reading notebooks
  "IProgress",
  "jupyter",
  "ipywidgets"
)

# Function to install or upgrade Python packages
install_or_upgrade <- function(pkg) {
  if (!py_module_available(pkg)) {
    # Install the package if not already available
    py_install(pkg, envname = "r-reticulate", pip = TRUE)
    message(paste(pkg, "has been installed."))
  } else {
    # Upgrade the package if already installed
    system2(
      command = "pip",
      args = c("install", "--upgrade", pkg),
      stdout = TRUE, stderr = TRUE
    )
    message(paste(pkg, "has been upgraded to the latest version."))
  }
}

# Install or upgrade all listed packages
for (pkg in pkgs) {
  install_or_upgrade(pkg)
}

message("All specified packages are installed or upgraded.")


numpy has been upgraded to the latest version.

pandas has been upgraded to the latest version.

streamlit has been upgraded to the latest version.



Using virtual environment 'r-reticulate' ...


+ /home/resurreccion_cmc_gmail_com/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user python_dateutil

python_dateutil has been installed.

tabulate has been upgraded to the latest version.

swifter has been upgraded to the latest version.

rpy2 has been upgraded to the latest version.

pyreadr has been upgraded to the latest version.

papermill has been upgraded to the latest version.

nbformat has been upgraded to the latest version.

IProgress has been upgraded to the latest version.

jupyter has been upgraded to the latest version.

ipywidgets has been upgraded to the latest version.

All specified packages are installed or upgraded.



# Libraries

In [2]:
# Update submodules like the grouper
system("git submodule update --init --recursive")

# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
gc()

# List, install (if applicable), and load packages
## Required packages
required_packages <- c(
  "data.table", "here", "tictoc", "stringr", "stringi", "lubridate",
  "profvis", "hash", "future", "future.apply", "knitr", "htmlwidgets",
  "parallelly", "stringdist", "parallel", "reticulate", "bigrquery",
  "jsonlite", "googleCloudStorageR", "haven", "fst"
  # , "docstring", "progress" # Comma is here so if I uncomment this line it
  # automatically works without having to type or delete a comma after haven
)
# Function to install and load packages quietly
install_and_load <- function(package) {
  if (!require(package, character.only = TRUE, quietly = TRUE)) {
    install.packages(package, dependencies = TRUE, quiet = TRUE)
  }
  suppressPackageStartupMessages(library(package, character.only = TRUE))
}

# Apply the function to each required package
invisible(lapply(required_packages, install_and_load))


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1793029,95.8,2873235,153.5,2873235,153.5
Vcells,3186705,24.4,8388608,64.0,5165780,39.5


here() starts at /home/resurreccion_cmc_gmail_com/drg-pipeline


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift



Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy




# Parameters

Change which year to process in 
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change other rarely touched parameters in 
`~/drg-pipeline/data-cleaning/r_scripts_v2/00_v2_params-fpaths.R`

In [3]:
# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample <- FALSE
# TODO: Add description here
to_write <- TRUE
# TODO: Add description here
to_flush <- FALSE
# TODO: Add description here
to_parallel <- as.logical(Sys.getenv("TO_PARALLEL", "TRUE"))
to_parallel <- TRUE
# TODO: Add description here
to_debug <- FALSE

# detect available threads
nthreads <- parallelly::availableCores()


# R Scripts

In [4]:
scripts_path <- here("data-cleaning/r_scripts_v2")

# List all R files in the directory with full paths, sorted by filename
r_files <- list.files(scripts_path, pattern = "\\.R$", full.names = TRUE)

# Source each file sequentially
for (file in r_files) {
  message(Sys.time(), " Sourcing: ", file)
  source(file)
}


2024-10-30 08:20:53.296678 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R

All directories exist.


Total Rows via cached object: 12757064

Utilizing 4 cores (8 threads)


2024-10-30 08:20:53.844527 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/0.2.0.process_helper_functions.R

2024-10-30 08:20:53.846634 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/0.3.0.summary_helper_functions.R

2024-10-30 08:20:53.848733 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/1.0.query_bq_to_dt.R

2024-10-30 08:20:53.85037 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/2.0.split_and_save_part.R

2024-10-30 08:20:53.85214 Sourcing: /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/r_scripts_v2/3.0.ensure_sample_files_exist.R

2024-10-30 08:20:53.853724 Sourcing: /home/resurreccion_cmc_gmail_com/dr

# Load Full Claims from GCS

In [5]:
# Load raw claims from GCS only if they don't exist on the VM yet
for (year in 2018:2023) {
  # Assign the correct file extension based on the year
  file_type <- if (year %in% c(2022:2023)) ".tsv" else ".csv"
  file_name <- paste0(full_claims_prefix, year, file_type)
  bq_name <- paste0(full_claims_bq_prefix, year, file_type)

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "drg-pipeline") {
      system(
        paste0(
          "cd .. && gsutil cp gs://phic-claims-raw/",
          bq_name, " ", raw_claims_path
        ),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not drg-pipeline")
    }
  } else {
    message(paste(
      "File", file_name,
      "already exists in the target directory. Skipping download.\n"
    ))
  }
}


File claims_extract_CLAIMS 2018.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2019.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2020.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2021.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2022.tsv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2023.tsv already exists in the target directory. Skipping download.




# Load Mapping Data

In [6]:
to_print_mapping_data <- FALSE

# Helper function to print all rows of data.tables
if (to_print_mapping_data) {
  print_all <- function(dt, title) {
    cat("\n---", title, "---\n")
    print(dt, nrow = Inf) # Print all rows
  }
}

# 1. Query and load `grouper_v5.proc`
message("Querying and loading `grouper_v5.proc`...")
proc_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.proc`")
proc <- query_bq_to_dt(proc_query)
proc[, CODE := as.character(CODE)]
if (to_print_mapping_data) print_all(proc, "Loaded `grouper_v5.proc`")

# 2. Query and load `phic.acr_rvs_map`
message("Querying and loading `phic.acr_rvs_map`...")
rvs_icd9_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_rvs_map`")
rvs_icd9 <- query_bq_to_dt(rvs_icd9_query)

# Convert RVS to character and handle ICD9CM conversion
message("Processing `phic.acr_rvs_map`...")
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]
if (to_print_mapping_data) print_all(rvs_icd9, "Processed `phic.acr_rvs_map`")

# Merge with proc to classify by DRGUSE
message("Merging `rvs_icd9` with `grouper_v5.proc`...")
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]
if (to_print_mapping_data) print_all(rvs_icd9, "Merged `rvs_icd9`")

# 3. Query and load `phic.acr_procedure`
message("Querying and loading `phic.acr_procedure`...")
acr_rvs_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_procedure`")
acr_rvs <- query_bq_to_dt(acr_rvs_query)
if (to_print_mapping_data) print_all(acr_rvs, "Loaded `phic.acr_procedure`")

# 4. Query and load `grouper_v5.i10`
message("Querying and loading `grouper_v5.i10`...")
i10_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10`")
tdrg_icd10 <- query_bq_to_dt(i10_query)
setkey(tdrg_icd10, "CODE")
if (to_print_mapping_data) print_all(tdrg_icd10, "Loaded `grouper_v5.i10`")

# Subset and assign to acc_pdx
message("Extracting ACCPDX codes from `grouper_v5.i10`...")
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])
if (to_print_mapping_data) print_all(acc_pdx, "Extracted ACCPDX Codes")
acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_pdx) {
  assign(code, TRUE, envir = acc_pdx_env)
}

# 5. Query and load `icd.phl_icd10`
message("Querying and loading `icd.phl_icd10`...")
phl_icd10_query <- paste0("SELECT * FROM `", gcp_proj, ".icd.phl_icd10`")
phl_icd10 <- query_bq_to_dt(phl_icd10_query)
if (to_print_mapping_data) print_all(phl_icd10, "Loaded `icd.phl_icd10`")

# Filter and process neoplasms
message("Processing neoplasm ICD-10 codes...")
neoplasms_dt_actual <- as.data.table(phl_icd10[
  grepl("/", icd10), .(icd10)
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])
if (to_print_mapping_data) print_all(neoplasms_dt_actual, "Processed Neoplasm ICD-10 Codes")

# 6. Query and load `drg-pipeline.grouper_v5.i10vx`
message("Querying and loading `grouper_v5.i10vx`...")
i10vx_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10vx`")
i10vx <- query_bq_to_dt(i10vx_query)
setkey(i10vx, "code")
acc_icd <- unique(i10vx[, code])
if (to_print_mapping_data) print_all(i10vx, "Loaded `grouper_v5.i10vx`")
if (to_print_mapping_data) print_all(acc_icd, "Extracted ACC_ICD Codes")

# 7. Query and load `drg-pipeline.hci.temp_hci`
message("Querying and loading `hci.temp_hci`...")
hci_query <- paste0("SELECT * FROM `", gcp_proj, ".hci.temp_hci`")
hci <- query_bq_to_dt(hci_query)
if (to_print_mapping_data) print_all(hci, "Loaded `hci.temp_hci`")


Querying and loading `grouper_v5.proc`...

Querying and loading `phic.acr_rvs_map`...

Processing `phic.acr_rvs_map`...

Merging `rvs_icd9` with `grouper_v5.proc`...

Querying and loading `phic.acr_procedure`...

Querying and loading `grouper_v5.i10`...

Extracting ACCPDX codes from `grouper_v5.i10`...

Querying and loading `icd.phl_icd10`...

Processing neoplasm ICD-10 codes...

Querying and loading `grouper_v5.i10vx`...

Querying and loading `hci.temp_hci`...



Test functions

In [7]:
# map_icd10 <- function(c1, c2, clin_icd,
#                       thai_icd10 = tdrg_icd10,
#                       covidrvs = covid_rvs,
#                       neoplasmsdtactual = neoplasms_dt_actual,
#                       acrrvs = acr_rvs) {
#   # Helper: Trim numeric suffixes (e.g., J1892 -> J189)
#   trim_code <- function(code) {
#     sub("(\\D+\\d{3})(\\d*)$", "\\1", code)
#   }

#   # Helper: Check if a code exists in a valid set
#   code_exists <- function(code, valid_set) {
#     code %in% valid_set
#   }

#   # Prepare sets of valid codes from datasets
#   valid_codes <- unique(thai_icd10$CODE)
#   neoplasm_codes <- unique(neoplasmsdtactual$icd10)
#   rvs_codes <- unique(acrrvs$rvs)

#   # Generate ICD-10 mapping
#   generate_icd10_mapping <- function(icds) {
#     icd_mapping <- list()
#     modified_count <- 0

#     for (code in icds) {
#       # 1. **Check for an exact match** with the original code
#       if (code_exists(code, valid_codes)) {
#         icd_mapping[[code]] <- list(match_type = "Exact", original = code, mapped = code)
#         next
#       }

#       # **Skip invalid, NA, or procedural/neoplasm codes**
#       if (is.na(code) || grepl("^[0-9]", code) || grepl("^[A-Z]{2,}", code) ||
#         code_exists(code, neoplasm_codes) || code_exists(code, rvs_codes)) {
#         next
#       }

#       # 2. **Try adding '9'** for 3-character codes
#       if (nchar(code) == 3) {
#         modified_code <- paste0(code, "9")
#         if (code_exists(modified_code, valid_codes)) {
#           icd_mapping[[code]] <- list(match_type = "Modified", original = code, mapped = modified_code)
#           modified_count <- modified_count + 1
#           next
#         }
#       }

#       # 3. **If no match, attempt trimming** (regardless of length)
#       trimmed_match <- FALSE
#       trimmed_code <- trim_code(code) # Trim code for further checks
#       if (nchar(trimmed_code) >= 3) { # Ensure trimmed code is valid length
#         if (code_exists(trimmed_code, valid_codes)) {
#           icd_mapping[[code]] <- list(match_type = "Modified", original = code, mapped = trimmed_code)
#           modified_count <- modified_count + 1
#           trimmed_match <- TRUE
#           next # Stop here if a trimmed match is found
#         }

#         # Try sequentially shorter partial codes if trimming fails
#         for (i in seq_len(nchar(trimmed_code) - 3)) {
#           partial_code <- substr(trimmed_code, 1, nchar(trimmed_code) - i)
#           if (code_exists(partial_code, valid_codes)) {
#             icd_mapping[[code]] <- list(match_type = "Modified", original = code, mapped = partial_code)
#             modified_count <- modified_count + 1
#             trimmed_match <- TRUE
#             break
#           }
#         }
#       }

#       # 4. **Mark as unmatched** if no match found
#       if (!trimmed_match) {
#         icd_mapping[[code]] <- list(match_type = "Unmatched", original = code, mapped = NA)
#       }
#     }

#     list(mapping = icd_mapping, modified_count = modified_count)
#   }

#   # Get all unique ICD codes, excluding COVID-related ones
#   icds <- setdiff(unique(unlist(c(c1, c2, clin_icd))), covidrvs)

#   # Generate the ICD-10 mapping
#   mapping_info <- generate_icd10_mapping(icds)
#   icd_mapping <- mapping_info$mapping

#   # Identify unmatched codes
#   unmatched_codes <- names(Filter(function(x) x$match_type == "Unmatched", icd_mapping))

#   # Create a data.table of unmatched codes by source
#   unmatched_sources <- rbindlist(lapply(c("c1", "c2", "clin_icd"), function(col_name) {
#     col_values <- get(col_name)
#     data.table(code = unlist(col_values), source = col_name)[, .(count = .N), by = .(code, source)]
#   }))
#   unmatched_sources <- unmatched_sources[code %in% unmatched_codes]

#   # Create ICD-10 mapping data.table for matches
#   icd10_map <- data.table(
#     phl_icd10 = names(icd_mapping), # Rename from original_code to phl_icd10
#     thai_icd10 = sapply(icd_mapping, `[[`, "mapped"), # Rename from mapped_code to thai_icd10
#     match_type = sapply(icd_mapping, `[[`, "match_type")
#   )

#   # Apply the mapping to input columns
#   apply_icd10_mapping <- function(codes) {
#     sapply(codes, function(code) {
#       if (!is.null(icd_mapping[[code]]) && !is.null(icd_mapping[[code]]$mapped)) {
#         icd_mapping[[code]]$mapped
#       } else {
#         code # Return the original code if no match found
#       }
#     })
#   }

#   # Apply the mapping
#   c1_mapped <- lapply(c1, apply_icd10_mapping)
#   c2_mapped <- lapply(c2, apply_icd10_mapping)
#   clin_icd_mapped <- lapply(clin_icd, apply_icd10_mapping)

#   # Return the results
#   list(
#     c1 = c1_mapped,
#     c2 = c2_mapped,
#     clin_icd = clin_icd_mapped,
#     icd10_map_dt = icd10_map,
#     unique_icds = icds,
#     unmatched_codes = unmatched_codes,
#     unmatched_sources = unmatched_sources,
#     icd_mapping_res = icd_mapping,
#     modified_count = mapping_info$modified_count,
#     valid_codes = valid_codes,
#     rvs_codes = rvs_codes,
#     neoplasm_codes = neoplasm_codes,
#     covid_rvs = covidrvs,
#     thai_icd10 = thai_icd10
#   )
# }

# # Step 1: Run the map_icd10 function
# result <- map_icd10(
#   c1 = list("A123", "J1892", "C50", "I109", "O82", "A971", "Z010A"),
#   c2 = list("D464", "A1234", "K35", "J142", "Y95"),
#   clin_icd = list("B345", "N17", "I109", "C22", "A972", "E862"),
#   thai_icd10 = tdrg_icd10,
#   covidrvs = covid_rvs,
#   neoplasmsdtactual = neoplasms_dt_actual,
#   acrrvs = acr_rvs
# )

# # Step 2: Extract the results from map_icd10
# exact_matches <- Filter(function(x) x$match_type == "Exact", result$icd_mapping_res)
# modified_matches <- Filter(function(x) x$match_type == "Modified", result$icd_mapping_res)
# unmatched_codes <- Filter(function(x) x$match_type == "Unmatched", result$icd_mapping_res)

# # Step 3: Print the summary of matches
# cat("--- Match Summary ---\n")
# cat("Exact Matches Count: ", length(exact_matches), "\n")
# cat("Modified Matches Count: ", length(modified_matches), "\n")
# cat("Unmatched Codes Count: ", length(unmatched_codes), "\n\n")

# # Step 4: Print detailed outputs for each match type
# cat("--- Exact Matches ---\n")
# print(data.table(
#   code = names(exact_matches),
#   original = sapply(exact_matches, `[[`, "original"),
#   mapped = sapply(exact_matches, `[[`, "mapped"),
#   match_type = "Exact"
# ))

# cat("\n--- Modified Matches ---\n")
# print(data.table(
#   code = names(modified_matches),
#   original = sapply(modified_matches, `[[`, "original"),
#   mapped = sapply(modified_matches, `[[`, "mapped"),
#   match_type = "Modified"
# ))

# cat("\n--- Unmatched Codes ---\n")
# print(data.table(
#   code = names(unmatched_codes),
#   original = sapply(unmatched_codes, `[[`, "original"),
#   mapped = sapply(unmatched_codes, `[[`, "mapped"),
#   match_type = "Unmatched"
# ))


# Data Cleaning Proper

Part 1: File Read In & Partial File Creation

In [8]:
# Step 1: Read the header of the full claims file
full_header <<- fread(
  file = full_claims_file,
  nrows = 1, colClasses = "character",
  header = TRUE
)

partial_file <<- here(raw_claims_parts_path, paste0(
  full_claims_prefix, year_to_load,
  "_part_", sprintf("%02d", split_parts),
  "_of_", split_parts, ".rds"
))

# Step 2: Check if the split part file already exists.
# If not, read the full claims file.
if (!file.exists(partial_file)) {
  # Read the full file into memory
  full_file <<- fread(
    file = full_claims_file, colClasses = "character",
    header = TRUE
  )
}

# Step 3: Split the file into parts and save them
start_time <- Sys.time() # Record start time
for (split_loop_part in 1:split_parts) split_and_save_part(split_loop_part)

# Step 4: DEPRECATED


Part 2: Main Data Cleaning Loop

In [9]:
# Step 5: Loop through each part and process the partial files
# saveWidget(profvis({
for (loop_part in 1:split_parts) {
  # loop_part <- 1
  start_time <- Sys.time() # Record start time for processing
  partial_claims_file <<- here(raw_claims_parts_path, paste0(
    full_claims_prefix, year_to_load,
    "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
  ))

  # Step 6: Handle sampling logic if applicable
  if (to_sample) {
    sampled_claims_file <<- here(raw_claims_samples_path, paste0(
      "sampled_claims_", year_to_load, "_", sample_size,
      "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
    ))

    ensure_sample_files_exist(loop_part) # Ensure sample files exist
  }

  # Step 7: Read the appropriate file (sample or full)

  read_result <- read_appropriate_file(loop_part)
  # The data to process
  read_in_dt <- read_result$read_result_dt
  # Any replacements summary
  read_in_replacement_summary <- read_result$read_result_replacement_summary

  # Step 8: Split the data into chunks for parallel processing
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(
    read_in_dt,
    rep(
      1:nthreads,
      each = chunk_size,
      length.out = nrow(read_in_dt)
    )
  )

  # Step 9: Apply parallel processing
  # See function(s) before the loop
  if (to_debug) {
    parallel_results <- process_chunk(chunks[[1]])
  } else if (to_parallel) {
    parallel_results <- mclapply(
      chunks, process_chunk,
      mc.cores = nthreads
    )
  } else {
    parallel_results <- lapply(
      chunks, process_chunk
    )
  }

  rbound_dt <- rbindlist(lapply(
    parallel_results,
    function(res) {
      res$return_chunk
    }
  ))

  # Step 11: Check for invalid primary diagnoses (PDx) and update the summary
  invalid_pdx_indices <- which(
    !is.na(rbound_dt$pdx) & rbound_dt$pdx != "" &
      !sapply(rbound_dt$pdx, function(x) exists(x, acc_pdx_env))
  )
  if (length(invalid_pdx_indices) > 0) {
    message(paste("Invalid PDx found:", rbound_dt$pdx[invalid_pdx_indices]))
    pdx_success_list[[loop_part]] <- FALSE
  } else {
    pdx_success_list[[loop_part]] <- TRUE
  }

  for (i in seq_along(parallel_results)) {
    parallel_results[[i]]$return_summary$pdx_success <- pdx_success_list[[loop_part]]
    parallel_results[[i]]$return_summary$replacement_summary <- read_in_replacement_summary
  }

  # Step 10: Combine results from all parallel chunks
  parallel_summaries <- lapply(
    parallel_results,
    function(res) {
      res$return_summary
    }
  )

  summarized_dt <- rbound_dt # Store the summarized data
  combined_chunk_summary[[loop_part]] <- parallel_summaries

  # Step 12: Write processed data to checkpoint file if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 13: Collect summaries for each part
  all_parts_summaries[[loop_part]] <- combined_chunk_summary[[loop_part]]
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(),
    start_time,
    units = "secs"
  ))

  # Step 14: Update status and ETA
  print_status_update(loop_part, split_parts, processing_times, "clean")

  if (loop_part == 1) dim_dt <- dim(summarized_dt)

  nrow_end[[loop_part]] <- nrow(summarized_dt)
  # Step 15: Clean up memory after processing each part
  rm(read_in_dt, rbound_dt, summarized_dt)
  gc()
}
# }), profvis_fpath)


Finished cleaning 15 of 15 parts in 1h 30m 45s (ETA 0s)            

Part 3: Merge Partial Outputs & Print Checks and Summaries

In [10]:
tryCatch(
  {
    # Step 1: Validate row counts across parts
    for (nrow_part in 1:split_parts) {
      if (nrow_start[[nrow_part]] != nrow_end[[nrow_part]]) {
        warning(
          "WARNING: Row Count Mismatch! Part ", nrow_part,
          " has ", nrow_start[[nrow_part]], " starting rows and ",
          nrow_end[[nrow_part]], " ending rows\n"
        )
        stop("ERROR: Row Count Mismatch")
      }
    }
    message("\nRow Counts Match for All Parts\n")

    # Step 2: Combine all parts into a master data table
    master_dt_list <- lapply(1:split_parts, function(read_part) {
      readRDS(here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
      )))
    })

    master_dt <- rbindlist(master_dt_list, fill = TRUE)
    rm(master_dt_list)
    gc()

    # Step 3: Save the combined master data table
    if (to_write) {
      saveRDS(master_dt, here(
        checkpoint_2_path, paste0(
          checkpoint_2_prefix, year_to_load, suffix, ".rds"
        )
      ), compress = TRUE)
    }
  },
  error = function(e) {
    message("Error encountered. Loading the previously saved master data table.")
    master_dt <- readRDS(here(
      checkpoint_2_path, paste0(
        checkpoint_2_prefix, year_to_load, suffix, ".rds"
      )
    ))
  }
)

print_summary_tables(aggregate_all_summaries(all_parts_summaries))



Row Counts Match for All Parts





Rename Success:
 TRUE 


Table: ICD Normalized Text for clin c1 & c2 Before Splitting
(Note: differences of only one period symbol are ignored)

|old_code       |new_code    | count|
|:--------------|:-----------|-----:|
|J91*           |J91         |    13|
|J18.99, Y95    |J1899Y95    |    62|
|D63.8*         |D638        |    43|
|I67.9+G46.8*   |I679G468    |     1|
|D63.0*         |D630        |    10|
|I63.9+G46.7*   |I639G467    |     7|
|E11.2+ N08.3*  |E112N083    |     8|
|N/A            |NA          |     4|
|c19t2          |C19T2       |     5|
|A09.9 E86.1    |A099E861    |     1|
|A02.2+ J17.0*  |A022J170    |     2|
|I67.9+G46.7*   |I679G467    |     2|
|c19t1          |C19T1       |     6|
|A01.0+ J17.0*  |A010J170    |     2|
|A09.9,E86.1    |A099E861    |     2|
|E14.2+ N08.3*  |E142N083    |     3|
|E10.2+ N08.3*  |E102N083    |     1|
|K74.6+I98.2*   |K746I982    |     1|
|E11.2+N08.3*   |E112N083    |     1|
|D63.8*         |D638        |    33|
|J91*           |J

Part 4: Save Output as .RDS

In [11]:
if (exists("master_dt")) {
  result <- data.table::copy(master_dt)
  rm(master_dt)
  gc()
} else {
  result <- readRDS(here(
    checkpoint_2_path, paste0(
      checkpoint_2_prefix, year_to_load, suffix, ".rds"
    )
  ))
  gc()
}

saveRDS(result, here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
), compress = TRUE)


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,102649510,5482.1,342178400,18274.3,179192319,9570.0
Vcells,733359373,5595.1,1626660759,12410.5,1355483966,10341.6


# Post Cleaning Steps

Part 1: Stata Modifications

In [12]:
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))

# Convert both id_hci and PMCC_NO to characters
# to ensure consistency for joining
result[, id_hci := as.character(id_hci)]
hci[, PMCC_NO := as.character(PMCC_NO)]

# Remove leading zeros from numeric PMCC_NO values,
# while preserving non-numeric values
hci[, PMCC_NO_stripped := fifelse(
  grepl("^[0-9]+$", PMCC_NO),
  sub("^0+", "", PMCC_NO), # Remove leading zeros from numeric strings
  PMCC_NO # Keep non-numeric values unchanged
)]

# Keep only necessary columns from hci for the join
hci_subset <- hci[, .(
  PMCC_NO_stripped,
  SOC_SECTOR, INST_NAME, CAT_24,
  REGION_NAME, PROVINCE_NAME
)]

# Perform a left join, keeping all rows in
# result and only matching rows from hci
result <- merge(
  result,
  hci_subset,
  by.x = "id_hci",
  by.y = "PMCC_NO_stripped",
  all.x = TRUE, # Keep all rows from result
  all.y = FALSE # Only include matching rows from hci
)

# ----- Filter rows by allowed CAT_24 categories -----
allowed_categories <- c(
  "INFIRMARY/DISPENSARY",
  "LEVEL 1 HOSPITAL", "LEVEL 2 HOSPITAL", "LEVEL 3 HOSPITAL"
)

# Count rows where CAT_24 is not in allowed categories
count_excluded <- result[
  !(CAT_24 %in% allowed_categories), .N
]
cat(
  "Number of rows where CAT_24 is not in the allowed categories:",
  count_excluded, "\n"
)


# Drop rows where CAT_24 is not in the allowed categories
result <- result[CAT_24 %in% allowed_categories]
# ----- Filter rows where clin_outpatient is TRUE -----
# Count rows where clin_outpatient is TRUE
count_clin_outpatient_true <- result[
  clin_outpatient == TRUE, .N
]
cat(
  "Number of rows where clin_outpatient is TRUE:",
  count_clin_outpatient_true, "\n"
)

# Drop rows where clin_outpatient is TRUE
result <- result[clin_outpatient != TRUE]
# ----- Filter rows where claim_status is not "G" -----
# Count rows where claim_status is not "G"
count_claim_status_not_g <- result[
  claim_status != "G", .N
]
cat(
  "Number of rows where claim_status is not 'G':",
  count_claim_status_not_g, "\n"
)

# Drop rows where claim_status is not "G"
result <- result[claim_status == "G"]
# Convert 'c1' from a list of character vectors
# into a single concatenated string
result[, c1_spc := sapply(c1, function(x) {
  if (is.null(x) || all(is.na(x))) {
    return(NA_character_) # Return NA if the list is empty or all values are NA
  } else {
    # Sort, remove duplicates, and concatenate
    return(paste(sort(unique(x)), collapse = ","))
  }
})]

# ----- Handle duplicate rows based on specific columns -----
duplicate_columns <- c(
  "id_pin", "pat_type", "pat_age",
  "pat_sex", "date_adm", "date_dis", "c1_spc", "claim_payout"
)

# Count duplicate rows based on the specified columns
count_duplicates <- result[
  duplicated(result[, ..duplicate_columns]), .N
]
cat(
  "Number of duplicated rows based on specified columns:",
  count_duplicates, "\n"
)

# Drop duplicate rows, keeping only the first occurrence
result <- result[!duplicated(result[, ..duplicate_columns])]
# ----- Remove unnecessary columns and reorder remaining columns -----
# Remove unnecessary columns
result[, c("c1", "c1_spc", "c2", "clin_rvs") := NULL]

# Perform garbage collection to free up memory
gc()

# Set the column order to a specified structure
setcolorder(result, c(
  "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm",
  "time_adm", "date_dis", "time_dis", "date_rec", "date_ref",
  "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
  "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent", "pat_memcat_child",
  "is_covid", "claim_status", "claim_payout", "claim_charge", "clin_discharge",
  "clin_outpatient", "clin_emergency", "clin_acc", "clin_c1", "clin_c2",
  "clin_sdx", "clin_proc", "clin_pdx", "clin_pdx_source", "SOC_SECTOR",
  "CAT_24", "INST_NAME", "REGION_NAME", "PROVINCE_NAME"
))

saveRDS(
  result,
  here(
    checkpoint_2_path,
    paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")
  ),
  compress = TRUE
)


Number of rows where CAT_24 is not in the allowed categories: 3172829 
Number of rows where clin_outpatient is TRUE: 4742780 
Number of rows where claim_status is not 'G': 227577 
Number of duplicated rows based on specified columns: 217 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,30497041,1628.8,218994176,11695.6,342178400,18274.3
Vcells,343301965,2619.2,1041062887,7942.7,1626660756,12410.5


# Part 2: Final Checks

Part 2A: Final Checks - Checking for Exponential Form Numbers

In [13]:
print(result[grepl("e", id_series)])
print(result[grepl("e", id_pin)])
print(result[grepl("e", id_hci)])
# Search for rows where any element in clin_sdx is "A"
result[, if (any(sapply(
  clin_sdx,
  function(row) "A" %in% row
))) {
  print(.SD)
}, by = seq_len(nrow(result))]


Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...
Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...
Key: <id_hci>
Empty data.table (0 rows and 41 cols): id_year,id_series,id_pin,id_hci,id_hcp,date_adm...


seq_len
<int>


Part 2B: Final Checks - Checking for Dates Before 1900

In [14]:
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))
date_cols <- c(
  "date_adm", "date_dis", "date_rec", "date_ref",
  "date_check", "pat_bdate", "date_ext"
)

# Find rows where any date column has a date before 1900-01-01
rows_with_old_dates <- result[Reduce(`|`, lapply(
  .SD,
  function(x) x < as.Date("1900-01-01")
)), .SDcols = date_cols]

# Print the resulting rows
print(rows_with_old_dates)


Null data.table (0 rows and 0 cols)


Part 2C: Final Checks - Dropping Duplicate Columns

In [15]:
result <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))
result[, c("c1", "c2", "clin_rvs") := NULL]


Part 2D: Final Checks - Checking Age Changes

In [16]:
before <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds")
))
after <- readRDS(here(
  checkpoint_2_path,
  paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")
))

# Ensure both data.tables have the same key columns for comparison
setkey(before, id_series)
setkey(after, id_series)

# Identify rows where pat_age is different
# between the two tables, handling NA values
pat_age_diff_na <- before[after,
  on = .(id_series), nomatch = 0,
  # Explicitly name pat_age_before as coming from "before"
  .(id_series, pat_bdate,
    pat_age_before = x.pat_age,
    pat_age_after = i.pat_age
  ),
  by = .EACHI
]

# Filter to show rows where one value is NA and
# the other is not or the values are simply different
pat_age_diff_na <- pat_age_diff_na[
  (is.na(pat_age_before) & !is.na(pat_age_after)) |
    (!is.na(pat_age_before) & is.na(pat_age_after)) |
    (pat_age_before != pat_age_after)
]

# Print the differences
cat("Rows where pat_age is NA in one table but
not in the other, or where the values differ:\n")
print(pat_age_diff_na)


Rows where pat_age is NA in one table but
not in the other, or where the values differ:
Key: <id_series>
Empty data.table (0 rows and 5 cols): id_series,id_series,pat_bdate,pat_age_before,pat_age_after


# BQ Upload

In [17]:
if (nrow(result) == total_rows) bq_table <- paste0("claims_", year_to_load)

# Check if the table should be dropped and replaced
tryCatch(
  {
    bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
    message("Table dropped successfully.\n")
  },
  error = function(e) {
    # If the table does not exist, just continue
    if (grepl("Not found", e, ignore.case = TRUE)) {
      message("Table does not exist, nothing to drop.\n")
    } else {
      # If it's a different error, re-throw the error
      stop(e)
    }
  }
)

# Attempt to create the table
tryCatch(
  {
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here(
        "data-cleaning/r_scripts_v2",
        "bq_schema_cleaning.json"
      ), simplifyDataFrame = FALSE)
    )
    message("Table created successfully.\n")
  },
  error = function(e) {
    # Check if the error message indicates that the table already exists
    if (grepl("already exists", e, ignore.case = TRUE)) {
      message("Table already exists. Skipping creation and upload.")
    } else {
      # If it's a different error, re-throw the error
      stop(e)
    }
  }
)

# Upload to BQ only if table is empty
if (to_write) {
  chunk_size <- 1000000 # Adjust the chunk size based on memory availability
  num_chunks <- ceiling(nrow(result) / chunk_size)

  for (i in seq_len(num_chunks)) {
    chunk <- result[
      ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)),
    ]

    bq_table_upload(
      bq_table(gcp_proj, bq_dataset, bq_table),
      values = chunk,
      write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
    )
  }
}


Auto-refreshing stale OAuth token.

Table dropped successfully.


Table created successfully.




# Debug Steps

In [18]:
concatenate_r_files <- function(input_path, output_file) {
  # List all .R files in the directory
  r_files <- list.files(
    path = input_path,
    pattern = "\\.R$", full.names = TRUE
  )

  # Delete the existing output file if it exists
  if (file.exists(output_file)) {
    file.remove(output_file)
  }

  # Read and concatenate contents
  file_contents <- lapply(r_files, readLines)
  concatenated_content <- unlist(file_contents)

  # Write concatenated content to the output file
  cat(concatenated_content, file = output_file, sep = "\n")
}

# Consolidate all r_scripts scripts into debug.R; useful for debugging
concatenate_r_files(
  here::here("data-cleaning/r_scripts_v2"),
  here::here("data-cleaning/debug/r_scripts_v2.R")
)

system(
  paste(
    "cd ~/drg-pipeline &&",
    "jupyter nbconvert",
    "--no-prompt",
    "--to script data-cleaning/drg-cleaning-v2.ipynb",
    "--output debug/drg-cleaning-v2"
  )
)


# Clean-up Steps

In [19]:
# Assign values using regular assignment (no need for <<- if declared globally)
if (TRUE) {
  to_debug <- TRUE
  to_flush_master <- FALSE
  to_flush_partial <- FALSE
}

# Define the paths and their corresponding conditions
paths <- list(
  to_flush_master = c(
    "data-cleaning/cache",
    "data-cleaning/data/profvis",
    "data-cleaning/data/aux-files",
    "data-cleaning/data/checkpoints",
    "data-cleaning/debug"
  ),
  to_flush_partial = c(
    "data-cleaning/data/claims/raw/parts",
    "data-cleaning/data/claims/raw/samples"
  )
)

# Iterate over the paths and delete directories if the corresponding condition is true
for (condition in names(paths)) {
  if (get(condition, envir = .GlobalEnv)) { # Ensure the variables are accessed in the global environment
    system(paste(
      "rm -r",
      paste(here::here(unlist(paths[[condition]])), collapse = " ")
    ))
  }
}

# Clean up the environment and run garbage collection if debugging is enabled
if (to_debug) {
  rm(list = ls(), envir = .GlobalEnv) # Ensure global environment is cleared
  gc()
}


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2493653,133.2,292270436,15609.0,365338045,19511.2
Vcells,21179173,161.6,2100298735,16024.1,2625373418,20030.1
